# Companion — Setup & Test

Run top to bottom. Each stage confirms one subsystem before the next stage uses it — so failures are localised instantly.

Select the **companion_env** kernel in the top-right, or activate it in a terminal:
```bash
source companion_env/bin/activate
```

## Stage 0 — Install

**Run `scripts/setup.sh` in a terminal first** (it needs sudo — Jupyter can't prompt). It installs system packages, CUDA deps, the CH341 serial driver for the ESP32 screen, builds llama.cpp with CUDA, and sets up the venv.

In [ ]:
import sys, os
os.environ['PATH'] = os.path.dirname(sys.executable) + ':' + os.environ.get('PATH', '')
print('Python:', sys.executable)

# If you haven't run `bash scripts/setup.sh` in a terminal yet, stop now and do that.
# (It needs sudo access for system packages + the CH341 kernel driver.)
!pip install -r requirements.txt

### Download every model

LLM (Gemma 4 E2B + E4B), VLM (Moondream), tool router, STT (Parakeet), TTS (Kokoro + Piper), vision (YuNet + HSEmotion), EOU, speaker ID. Idempotent — re-run any time.

In [ ]:
!python3 scripts/download_models.py

### Flash the ESP32 face firmware (once)

With the Diymore 2.8" screen plugged into a USB-C port on the Jetson.

In [ ]:
!bash scripts/flash_firmware.sh

## Stage 1 — Environment sanity

Confirms every library is importable, every model is on disk, and every device is detected.

In [ ]:
!python3 scripts/verify.py

## Stage 2 — Subsystems (unified debug GUI)

One window, one tab per subsystem — everything you used to launch separately is now in here. Run the cell below and walk through the tabs:

| Tab | What it verifies |
| --- | --- |
| **Audio** | Mic waveform, RMS, VAD probability, ReSpeaker DOA (click *Start*) |
| **STT** | Record 5 s and transcribe via Parakeet (or Whisper fallback) |
| **LLM** | Load Gemma 4 E2B/E4B, chat, see tok/s |
| **TTS** | Speak a test sentence with Kokoro or Piper; real-time factor shown |
| **Vision** | Live camera + face box + emotion label + circumplex + FPS/latency |
| **VLM** | Grab a frame → ask Moondream a question; captured frame is shown |
| **Tools** | Route an utterance through FunctionGemma, see parsed tool call |
| **Memory** | Add / search / forget speaker-scoped memories (Mem0 + ChromaDB) |
| **Face display** | Drive the ESP32 screen (or HDMI pygame fallback) with presets |
| **Face tracking** | Live YOLO-pose camera feed + servo PID (sim motors by default) |

> Tip: **Vision** and **Face tracking** share one camera pipeline — run either alone or both concurrently, no need to stop one first.


In [ ]:
!python3 -m tests.debug_gui

## Stage 3 — Combined tests

Everything together. Subsystems must already pass above. The `all` target runs the sanity checks sequentially and prints a final summary.

In [ ]:
!python3 -m tests.cli all

## Stage 4 — Launch the full app

Main window comes up, face animates on the touchscreen (or HDMI fallback), speak with SPACE held down.

Run this in a terminal — notebooks don't host Qt windows well:
```bash
source companion_env/bin/activate
python3 main.py
```